In [22]:
from pyspark.sql import SparkSession # type: ignore

spark = SparkSession.builder \
    .appName("SparkCourse") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/home/jovyan/work/setup/spark-warehouse") \
    .config("spark.hadoop.javax.jdo.option.ConnectionURL",
            "jdbc:derby:/home/jovyan/work/metastore_db;create=true") \
    .config("spark.hadoop.javax.jdo.option.ConnectionDriverName",
            "org.apache.derby.jdbc.EmbeddedDriver") \
    .enableHiveSupport() \
    .getOrCreate()

print("Spark version:", spark.version)

Spark version: 3.5.0


Lateral Join
    Lateral join allows to query right dataframe for each row of the left dataframe.
    Lateral joins are especially useful when:
        You need per-parent Top-N child rows
        You want to invoke TVFs with arguments derived from each row

In [27]:
"""
Find the most recent booking for each member
+---------+----------+---------+---------------+-------------------+-----+
|member_id|first_name|last_name|  facility_name|         start_time|slots|
"""

spark.sql("""
    SELECT m.memid AS member_id, m.firstname AS first_name, m.surname AS last_name, b.facid AS facility_id, b.starttime, b.slots
    FROM spark_db.members m
    LEFT JOIN LATERAL(
        SELECT facid, slots
        FROM spark_db.bookings
        WHERE memid = m.memid
        ORDER BY starttime DESC
        LIMIT 1
    ) b ON true
    WHERE m.memid > 0
""").show()

AnalysisException: [UNSUPPORTED_SUBQUERY_EXPRESSION_CATEGORY.ACCESSING_OUTER_QUERY_COLUMN_IS_NOT_ALLOWED] Unsupported subquery expression: Accessing outer query column is not allowed in this locationFilter (memid#130 = outer(memid#40))
+- SubqueryAlias spark_catalog.spark_db.bookings
   +- Relation spark_catalog.spark_db.bookings[bookid#128,facid#129,memid#130,starttime#131,slots#132] parquet
.; line 5 pos 8;
'Project [memid#40 AS member_id#1406, firstname#42 AS first_name#1407, surname#41 AS last_name#1408, facid#129 AS facility_id#1409, 'b.starttime, slots#132]
+- Filter (memid#40 > 0)
   +- LateralJoin lateral-subquery#1405 [memid#40], LeftOuter, true
      :  +- SubqueryAlias b
      :     +- GlobalLimit 1
      :        +- LocalLimit 1
      :           +- Project [facid#129, slots#132]
      :              +- Sort [starttime#131 DESC NULLS LAST], true
      :                 +- Project [facid#129, slots#132, starttime#131]
      :                    +- Filter (memid#130 = outer(m